In [3]:
import pandas as pd
import geopandas as gpd
import osmnx as ox


# CONFIG
HOURS_PER_YEAR = 8760

CAPACITY_FACTORS = {
    "Wind": 0.32,               #Fraunhofer statistics for offshore wind in germany
    "Solare Strahlungsenergie": 0.12
}

BRANDENBURG = "Brandenburg"

PLANTS_FILE = "Stromerzeuger.csv"

# projected CRS for Europe / distance-based operations
PROJECTED_CRS = "EPSG:3035"
GEOGRAPHIC_CRS = "EPSG:4326"

# optional max matching distance in meters
MAX_DISTANCE_M = 30000


# HELPERS

def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )

def print_check(name, df):
    print(f"{name}: {len(df):,} rows")

def require_columns(df, cols, df_name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing columns: {missing}")

# 1. LOAD AND CLEAN PLANTS

plants = pd.read_csv(PLANTS_FILE, sep=";", low_memory=False)
plants.columns = plants.columns.str.strip()

require_columns(
    plants,
    [
        "MaStR-Nr. der Einheit",
        "Bundesland",
        "Energieträger",
        "Nettonennleistung der Einheit",
        "Koordinate: Breitengrad (WGS84)",
        "Koordinate: Längengrad (WGS84)"
    ],
    "plants"
)

plants = plants[
    (plants["Bundesland"] == BRANDENBURG) &
    (plants["Betriebs-Status"] == "In Betrieb") &
    (plants["Energieträger"].isin(["Wind", "Solare Strahlungsenergie"]))
].copy()

plants = plants.rename(columns={
    "MaStR-Nr. der Einheit": "plant_id",
    "Anzeige-Name der Einheit": "plant_name",
    "Energieträger": "technology",
    "Nettonennleistung der Einheit": "capacity_kw",
    "Postleitzahl": "plant_postcode",
    "Ort": "plant_ort",
    "Gemeinde": "plant_municipality",
    "Landkreis": "plant_district",
    "Koordinate: Breitengrad (WGS84)": "lat",
    "Koordinate: Längengrad (WGS84)": "lon",
    "MaStR-Nr. der Lokation": "plant_location_id"
})

plants["capacity_kw"] = clean_numeric(plants["capacity_kw"])
plants["capacity_mw"] = plants["capacity_kw"] / 1000
plants["lat"] = clean_numeric(plants["lat"])
plants["lon"] = clean_numeric(plants["lon"])

plants = plants.dropna(subset=["plant_id", "capacity_mw", "lat", "lon"]).copy()

plants_gdf = gpd.GeoDataFrame(
    plants,
    geometry=gpd.points_from_xy(plants["lon"], plants["lat"]),
    crs=GEOGRAPHIC_CRS
)

print_check("Brandenburg wind/solar plants", plants_gdf)


# 2. ESTIMATE PRODUCTION

plants_gdf["capacity_factor"] = plants_gdf["technology"].map(CAPACITY_FACTORS)
plants_gdf["energy_mwh"] = (
    plants_gdf["capacity_mw"] *
    HOURS_PER_YEAR *
    plants_gdf["capacity_factor"]
)

print("Plants without capacity factor:", plants_gdf["capacity_factor"].isna().sum())


# 3. OSM SUBSTATIONS

# You can also use ox.geocode_to_gdf("Brandenburg, Germany") if needed
substations = ox.features_from_place(
    "Brandenburg, Germany",
    tags={"power": "substation"}
).reset_index()

print_check("Raw OSM substations", substations)

# keep only rows with geometry
substations = substations[substations.geometry.notna()].copy()

# keep a useful set of attributes if present
candidate_cols = [
    "osmid",
    "name",
    "operator",
    "voltage",
    "substation",
    "power",
    "geometry"
]
existing_cols = [c for c in candidate_cols if c in substations.columns]
substations = substations[existing_cols].copy()

# assign fallback ID
if "osmid" not in substations.columns:
    substations["osmid"] = substations.index.astype(str)

substations = substations.rename(columns={
    "osmid": "substation_id",
    "name": "substation_name",
    "operator": "substation_operator",
    "voltage": "substation_voltage",
    "substation": "substation_type"
})

# convert polygons/lines to centroids
substations = substations.to_crs(PROJECTED_CRS)
substations["geometry"] = substations.geometry.centroid

print_check("OSM substations prepared", substations)


# 4. SPATIAL MATCH: PLANTS -> NEAREST SUBSTATION

plants_proj = plants_gdf.to_crs(PROJECTED_CRS)

plants_with_sub = gpd.sjoin_nearest(
    plants_proj,
    substations,
    how="left",
    distance_col="distance_to_substation_m"
)

print_check("Plants matched to nearest OSM substation", plants_with_sub)

# optional: drop clearly unrealistic matches
plants_with_sub = plants_with_sub[
    plants_with_sub["distance_to_substation_m"].isna() |
    (plants_with_sub["distance_to_substation_m"] <= MAX_DISTANCE_M)
].copy()

print_check("Plants after max-distance filter", plants_with_sub)


# 5. BUILD PLANT-LEVEL 

plants_output = plants_with_sub.to_crs(GEOGRAPHIC_CRS).copy()

plant_output_cols = [
    "plant_id",
    "plant_name",
    "technology",
    "capacity_kw",
    "capacity_mw",
    "capacity_factor",
    "energy_mwh",
    "plant_postcode",
    "plant_ort",
    "plant_municipality",
    "plant_district",
    "plant_location_id",
    "substation_id",
    "substation_name",
    "substation_operator",
    "substation_voltage",
    "substation_type",
    "distance_to_substation_m",
    "geometry"
]

plant_output_cols = [c for c in plant_output_cols if c in plants_output.columns]
plants_output = plants_output[plant_output_cols].copy()


# 6. BUILD SUBSTATION-LEVEL OUTPUT

summary = (
    plants_with_sub.dropna(subset=["substation_id"])
    .groupby(
        [
            "substation_id",
            "substation_name",
            "substation_operator",
            "substation_voltage",
            "substation_type"
        ],
        dropna=False
    )
    .agg(
        total_capacity_mw=("capacity_mw", "sum"),
        total_energy_mwh=("energy_mwh", "sum"),
        n_plants=("plant_id", "nunique"),
        n_wind=("technology", lambda s: (s == "Wind").sum()),
        n_solar=("technology", lambda s: (s == "Solare Strahlungsenergie").sum()),
        avg_distance_to_substation_m=("distance_to_substation_m", "mean"),
        max_distance_to_substation_m=("distance_to_substation_m", "max")
    )
    .reset_index()
)

print_check("Substation summary", summary)

# attach geometry back
substation_geom = substations[
    ["substation_id", "geometry"]
].drop_duplicates(subset=["substation_id"]).copy()

summary_gdf = summary.merge(
    substation_geom,
    on="substation_id",
    how="left"
)

summary_gdf = gpd.GeoDataFrame(
    summary_gdf,
    geometry="geometry",
    crs=PROJECTED_CRS
).to_crs(GEOGRAPHIC_CRS)


# 7. SAVE OUTPUTS

plants_output.to_file("plants_with_osm_substations.geojson", driver="GeoJSON")
summary_gdf.to_file("substations_energy_summary.geojson", driver="GeoJSON")

plants_output.drop(columns="geometry").to_csv("plants_with_osm_substations.csv", index=False)
summary_gdf.drop(columns="geometry").to_csv("substations_energy_summary.csv", index=False)

print("\nSaved:")
print("- plants_with_osm_substations.geojson")
print("- plants_with_osm_substations.csv")
print("- substations_energy_summary.geojson")
print("- substations_energy_summary.csv")

Brandenburg wind/solar plants: 3,166 rows
Plants without capacity factor: 0


/home/wii/anaconda3/envs/unetenv/lib/python3.12/site-packages/osmnx/_overpass.py:271: UserWarning: This area is 15 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


Raw OSM substations: 3,546 rows
OSM substations prepared: 3,546 rows
Plants matched to nearest OSM substation: 3,166 rows
Plants after max-distance filter: 3,166 rows
Substation summary: 402 rows

Saved:
- plants_with_osm_substations.geojson
- plants_with_osm_substations.csv
- substations_energy_summary.geojson
- substations_energy_summary.csv


In [6]:
import folium

def create_map(plants, substations):
    m = folium.Map(location=[52.5, 13.4], zoom_start=7)

    for _, row in substations.iterrows():
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=4,
            color="red"
        ).add_to(m)

    for _, row in plants.iterrows():
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=2,
            color="blue"
        ).add_to(m)

    m.save("outputs/map.html")